# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [63]:
!pip install librosa
!pip install panns-inference


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [64]:
!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [65]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [66]:
# select the audio feature and display top three
audios = r"C:\Users\Shrabani P\IronHack\Week8_Day1\Lab_Week8_Day1\lab-music-recommendations\data\miaow_16k.wav"
audios[:3]

'C:\\'

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [67]:
print(data)
print(data.features)

Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})
{'filename': Value('string'), 'fold': Value('int64'), 'target': Value('int64'), 'category': Value('string'), 'esc10': Value('bool'), 'src_file': Value('int64'), 'take': Value('string'), 'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None)}


In [68]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)

try:
    import torchcodec
    print("TorchCodec:", torchcodec.__version__)
except Exception as e:
    print("TorchCodec problem:", e)

Python: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
PyTorch: 2.13.0+cpu
TorchCodec problem: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.13.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to loa

In [69]:
!ffmpeg -version

ffmpeg version 8.1.2-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
built with gcc 16.1.0 (Rev2, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-liboapv --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxavs2 --enable-libxeve --enable-libxvid --enable-libaom --enable-libjxl --enabl

In [70]:
from datasets import Audio

data = data.cast_column("audio", Audio(decode=False))

In [71]:
print(data[0]["audio"])

{'bytes': b'RIFF\xcc\xba\x06\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00D\xac\x00\x00\x88X\x01\x00\x02\x00\x10\x00data\xa8\xba\x06\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [72]:
audios[:3]

'C:\\'

In [73]:
print("Dataset rows:", len(data))
print("First audio:", data[0]["audio"])

Dataset rows: 2000
First audio: {'bytes': b'RIFF\xcc\xba\x06\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00D\xac\x00\x00\x88X\x01\x00\x02\x00\x10\x00data\xa8\xba\x06\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [74]:
print("Number of audio records:", len(data["audio"]))

Number of audio records: 2000


In [75]:
print(type(data[0]["audio"]))


<class 'dict'>


In [76]:
print(data[0]["audio"].keys())

dict_keys(['bytes', 'path'])


In [77]:
import librosa

audios = []

for a in data["audio"]:
    audio_array, sr = librosa.load(r"C:\Users\Shrabani P\IronHack\Week8_Day1\Lab_Week8_Day1\lab-music-recommendations\data\miaow_16k.wav", sr=None)
    audios.append(audio_array)

In [78]:
len(audios)

2000

In [79]:
audios[:3]

[array([-0.00323486, -0.00170898,  0.00308228, ..., -0.01098633,
        -0.0128479 , -0.01364136], shape=(107698,), dtype=float32),
 array([-0.00323486, -0.00170898,  0.00308228, ..., -0.01098633,
        -0.0128479 , -0.01364136], shape=(107698,), dtype=float32),
 array([-0.00323486, -0.00170898,  0.00308228, ..., -0.01098633,
        -0.0128479 , -0.01364136], shape=(107698,), dtype=float32)]

In [80]:
print(data[0]["audio"]["path"])
print(data[1]["audio"]["path"])
print(data[2]["audio"]["path"])

None
None
None


In [81]:
## import numpy as np

# select only the audio data from the dataset and store in a numpy array
##audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [82]:
import os
import urllib.request

panns_dir = os.path.expanduser("~/panns_data")
os.makedirs(panns_dir, exist_ok=True)

labels_path = os.path.join(
    panns_dir,
    "class_labels_indices.csv"
)

url = "https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv"

urllib.request.urlretrieve(url, labels_path)

print("Saved to:", labels_path)
print("File exists:", os.path.exists(labels_path))

Saved to: C:\Users\Shrabani P/panns_data\class_labels_indices.csv
File exists: True


In [83]:
from panns_inference import AudioTagging

print("Import successful")

Import successful


In [84]:
import os
import urllib.request

panns_dir = os.path.expanduser("~/panns_data")
os.makedirs(panns_dir, exist_ok=True)

checkpoint_path = os.path.join(
    panns_dir,
    "Cnn14_mAP=0.431.pth"
)

url = "https://zenodo.org/records/3987831/files/Cnn14_mAP%3D0.431.pth?download=1"

print("Downloading PANNs checkpoint...")
urllib.request.urlretrieve(url, checkpoint_path)

print("Saved to:", checkpoint_path)
print("File exists:", os.path.exists(checkpoint_path))
print("File size:", os.path.getsize(checkpoint_path))

Saved to: C:\Users\Shrabani P/panns_data\Cnn14_mAP=0.431.pth
File exists: True
File size: 327428481


In [85]:
# Load Audio Embedding Model

from panns_inference import AudioTagging

# Load the model using CPU
model = AudioTagging(
    checkpoint_path=None,
    device='cpu'
)

Checkpoint path: C:\Users\Shrabani P/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [86]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='mps') # change device to cpu if a gpu is not available

Checkpoint path: C:\Users\Shrabani P/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [87]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [88]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [89]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [90]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [91]:
index_name = "audio-search-demo"

In [92]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

In [93]:
pc.list_indexes()

{'indexes': [{'dimension': 384,
              'host': 'question-answering-d76fs6t.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'question-answering',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}},
             {'dimension': 512,
              'host': 'ironstore-ai-v2-d76fs6t.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'ironstore-ai-v2',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}},
             {'dimension': 768,
              'host': 'abstractive-question-answering-d76fs6t.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'abstractive-question-answering',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state

In [94]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [95]:
import numpy as np
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    i_end = min(i + batch_size, len(audios))

    batch = audios[i:i_end]

    # Convert list of audio arrays into one NumPy array
    batch = np.stack(batch).astype(np.float32)

    # Generate embeddings
    _, emb = model.inference(batch)

    # Create unique IDs
    ids = [str(idx) for idx in range(i, i_end)]

    # Prepare vectors for Pinecone
    to_upsert = list(zip(ids, emb.tolist()))

    # Upload vectors
    index.upsert(vectors=to_upsert)

index.describe_index_stats()

100%|██████████| 32/32 [04:26<00:00,  8.33s/it]


{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

In [96]:
print(type(audios))
print(type(audios[0]))

print(len(audios))
print(len(audios[0]))
print(len(audios[1]))
print(len(audios[2]))

<class 'list'>
<class 'numpy.ndarray'>
2000
107698
107698
107698


In [97]:
import numpy as np
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    
    # End position of batch
    i_end = min(i + batch_size, len(audios))
    
    # Extract batch
    batch = audios[i:i_end]
    
    # IMPORTANT: convert list of arrays → single NumPy array
    batch = np.stack(batch).astype(np.float32)
    
    # Generate PANNs embeddings
    _, emb = model.inference(batch)
    
    # Create unique IDs
    ids = [str(idx) for idx in range(i, i_end)]
    
    # Prepare vectors for Pinecone
    to_upsert = list(zip(ids, emb.tolist()))
    
    # Insert vectors into Pinecone
    index.upsert(vectors=to_upsert)

# Check Pinecone index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 32/32 [03:38<00:00,  6.82s/it]


{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [98]:
from IPython.display import Audio, display

audio_num = 400

# Get waveform from the audios list
query_audio = audios[audio_num]

# Get category from dataset
category = data[audio_num]["category"]

print("Query Audio:", category)
print("Audio shape:", query_audio.shape)

display(Audio(query_audio, rate=44100))

Query Audio: car_horn
Audio shape: (107698,)


We have got the sound of a car horn. Let's generate an embedding for this sound.

In [99]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [100]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '1055', 'score': 1.00001431, 'values': []},
             {'id': '1086', 'score': 1.00001431, 'values': []},
             {'id': '1477', 'score': 1.00001431, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [101]:
from IPython.display import Audio, display

# Play the top 3 similar audios
for r in results["matches"][:3]:
    
    # Get Pinecone ID
    idx = int(r["id"])
    
    # Get audio waveform from our audios list
    a = audios[idx]
    
    # Get category
    category = data[idx]["category"]
    
    # Display information
    print(
        f"ID: {idx} | "
        f"Category: {category} | "
        f"Similarity: {r['score']:.4f}"
    )
    
    # Play audio
    display(Audio(a, rate=44100))

ID: 1055 | Category: dog | Similarity: 1.0000


ID: 1086 | Category: chainsaw | Similarity: 1.0000


ID: 1477 | Category: car_horn | Similarity: 1.0000


In [102]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = audios[idx]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [103]:
from IPython.display import Audio, display

def find_similar_audios(id):
    
    # -------------------------
    # Query audio
    # -------------------------
    print("Query Audio:")
    
    query_audio = audios[id]
    category = data[id]["category"]
    
    print(f"ID: {id} | Category: {category}")
    
    display(Audio(query_audio, rate=44100))
    
    
    # -------------------------
    # Query Pinecone
    # -------------------------
    result = index.query(
        id=str(id),
        top_k=6
    )
    
    
    # -------------------------
    # Similar audios
    # -------------------------
    print("\nSimilar Audios:")
    
    count = 0
    
    for r in result["matches"]:
        
        idx = int(r["id"])
        
        # Skip the query audio itself
        if idx == id:
            continue
        
        # Get audio from our NumPy audio list
        a = audios[idx]
        
        # Get category
        category = data[idx]["category"]
        
        # Print result information
        print(
            f"ID: {idx} | "
            f"Category: {category} | "
            f"Similarity: {r['score']:.4f}"
        )
        
        # Play similar audio
        display(Audio(a, rate=44100))
        
        count += 1
        
        # Stop after 5 similar audios
        if count == 5:
            break

In [104]:
find_similar_audios(1642)


Query Audio:
ID: 1642 | Category: chainsaw



Similar Audios:
ID: 1001 | Category: coughing | Similarity: 1.0000


ID: 1002 | Category: snoring | Similarity: 1.0000


ID: 1003 | Category: washing_machine | Similarity: 1.0000


ID: 1004 | Category: washing_machine | Similarity: 1.0000


ID: 1005 | Category: snoring | Similarity: 1.0000


Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [105]:
find_similar_audios(452)

Query Audio:
ID: 452 | Category: crow



Similar Audios:
ID: 1001 | Category: coughing | Similarity: 1.0000


ID: 1002 | Category: snoring | Similarity: 1.0000


ID: 1003 | Category: washing_machine | Similarity: 1.0000


ID: 1004 | Category: washing_machine | Similarity: 1.0000


ID: 1005 | Category: snoring | Similarity: 1.0000


And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [106]:
!wget https://storage.googleapis.com/audioset/miaow_16k.wav

'wget' is not recognized as an internal or external command,
operable program or batch file.


We can load the audio into a Numpy array as follows:

In [107]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [108]:
import numpy as np
from IPython.display import Audio, display

# 'a' is your query audio waveform
query_audio = a[None, :].astype(np.float32)

print("Query audio shape:", query_audio.shape)

# Generate embedding
_, xq = model.inference(query_audio)

print("Embedding shape:", xq.shape)

# Query Pinecone with ONE embedding vector
results = index.query(
    vector=xq[0].tolist(),
    top_k=3
)

# Play top 3 similar audios
for r in results["matches"]:
    
    idx = int(r["id"])
    
    # Get waveform from audios list
    similar_audio = audios[idx]
    
    # Get category
    category = data[idx]["category"]
    
    print(
        f"ID: {idx} | "
        f"Category: {category} | "
        f"Similarity: {r['score']:.4f}"
    )
    
    display(Audio(similar_audio, rate=44100))

Query audio shape: (1, 296843)
Embedding shape: (1, 2048)
ID: 1001 | Category: coughing | Similarity: 0.7192


ID: 1002 | Category: snoring | Similarity: 0.7192


ID: 999 | Category: breathing | Similarity: 0.7192


Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [109]:
pc.delete_index(index_name)